In [21]:
# declare libraries
import json
import geojson
import pathlib
import time
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely

import pprint as pp

from shapely.geometry import shape, mapping
from shapely.validation import explain_validity
from shapely.ops import unary_union

import zipfile
from zipfile import ZipFile
import rioxarray as rxr
import xarray as xr
# import rasterio
from rasterio.plot import show

import matplotlib.pyplot as plt

import requests
from requests.auth import HTTPBasicAuth
import planet
from planet import Session, DataClient, OrdersClient, Auth, Planet
from planet.auth import APIKeyAuth

import os
import sys

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# import itertools

sys.path.append("../utils")

import config
import aoi_filter_maker as aoi

# import pyproj
# from pyproj import CRS, Transformer

# bring in 2019 training geometries geojson for testing
for yr in range(2019,2024):
    with open(f'/capstone/wildfire_prep/data/training_geometries/training_geometries_{yr}.geojson', 'r') as file:
        globals()[f"training_{yr}"] = dict(geojson.load(file))["features"]


# buffer_2019_df = gpd.read_file("/capstone/wildfire_prep/data/training_geometries/training_geometries_2019.geojson")

# declare misc vars
crs = "EPSG:4326"
#crs = "EPSG:3310"

data_api_url = "https://api.planet.com/data/v1"
orders_api_url = 'https://api.planet.com/compute/ops/orders/v2' 

Set api key and authorize

In [4]:
planet_key = config.planet_api
PL_API_KEY = planet_key

# authentication
auth = HTTPBasicAuth(planet_key, "")

# check we are communicating with http properly
response = requests.get(data_api_url, auth=auth, timeout=999)
print(response)

# make function for pagenation
def p(data):
    print(json.dumps(data, indent = 2))

<Response [200]>


### Init session

Enables the use of certain functions.

In [5]:
# setup
session = requests.Session()

retries = Retry(total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

# authenticate
session.auth = (planet_key, "")

res = session.get(data_api_url)
res

<Response [200]>

### Make AOI filters

We will make test cases for both polygon limits, and vertices limits.

In [4]:
# choose training geometries dataset
geometries = training_2023

collection_2023 = aoi.make_aoi_geojson_collection(geometries, 40)

step: begin = 0, end = 40
step: begin = 40, end = 80
step: begin = 80, end = 120
step: begin = 120, end = 160
step: begin = 160, end = 200
step: begin = 200, end = 240
step: begin = 240, end = 280
step: begin = 280, end = 320
step: begin = 320, end = 360
step: begin = 360, end = 400
step: begin = 400, end = 440
step: begin = 440, end = 480
step: begin = 480, end = 520
step: begin = 520, end = 560
step: begin = 560, end = 600
step: begin = 600, end = 640
step: begin = 640, end = 680
step: begin = 680, end = 720
step: begin = 720, end = 760
step: begin = 760, end = 800
step: begin = 800, end = 840
step: begin = 840, end = 880
step: begin = 880, end = 920
step: begin = 920, end = 960
step: begin = 960, end = 1000
step: begin = 1000, end = 1040
step: begin = 1040, end = 1080
step: begin = 1080, end = 1120
step: begin = 1120, end = 1160
step: begin = 1160, end = 1200
step: begin = 1200, end = 1240
step: begin = 1240, end = 1280
step: begin = 1280, end = 1320
step: begin = 1320, end = 1360
s

### Declare functions
1. Making the order
2. Pinging the api for success/general order status

In [5]:
def place_order(request, auth, retry_counter, index, month):
    order_url = None

    # make order request
    response = session.post(
        orders_api_url, 
        data = json.dumps(request), 
        auth = auth, 
        headers = headers, 
        timeout = 999
        )


    # if there is a rate limiting problem
    if response.status_code == 429:
        while retry_counter < 5 and response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 60))  # set 60 secs for the waiting time
            print(f"Rate limit hit. Retrying after {retry_after} seconds...")

            time.sleep(retry_after) # wait for 60 secs
            retry_counter = retry_counter + 1 # increment the retry counter

            return place_order(request, auth, retry_counter) # try the function again
    
    elif response.status_code == 400:
        print(f"Status code: {response.status_code}")
        print(f"Status comments: {response.text}")
        print("Bad Request, cancelling this order...")
        print(f"Bad index: {index} in month: {month}")
        return None



    json_error_i = 0
    while json_error_i < 5:
        try:
            # get ids of scenes
            order_id = response.json()["id"]
            print(order_id)

            # construct the url of our order
            order_url = orders_api_url + '/' + order_id

            break
        except json.JSONDecodeError:
            print("JSON parsing failed, attempting again...")
            json_error_i += 1
        except Exception as e:
            print("Some other error occured. Attempting again...")
            json_error_i += 1
        
        if json_error_i >= 5:
            print(f"Retry limit reached. Ordering failed in the place_order function.")
            return None

    
    return order_url

In [6]:
def poll_for_success(order_url, auth, num_loops = 999): 
    i = 0
    while(i < num_loops): 

        # iterate
        i += 1

        # get order request
        r = requests.get(order_url, auth = auth, timeout=999)
        response = r.json()

        # grab current state
        state = response["orders"][0]["state"]
        print(state)

        # compare it to a variety of end states and print it
        end_states = ["success", "failed", "partial"]
        if state in end_states:
            print(f"End State: {state}")
            break

        # wait 30 secs
        time.sleep(30)



In [7]:
# get variable name as string
def var_name_as_str(var):
    for name, value in globals().items():
        if value is var:
            return name


### run whole stress test

* capturing...
    1. how long each test case takes
    2. if each test case is successful

##### Make iteration variables

In [8]:
month_nums = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "01"]
year_nums = ["2019", "2020", "2021", "2022", "2023"]

i = 0
for year in year_nums: 
    globals()[f"year_iter_{year}"] = [*[str(2019 + i)] * 12, str(2020 + i)]
    i += 1

##### Ordering workflow

In [10]:
"""REMEMBER TO SET CHUNKS AND YEAR ITER VARIABLES!!!!"""

chunks = collection_2023
# added the index for which to contine from which the collection last errored

# safety check, only run cell if you specifically say "yes"
response = input("Are you sure you want to execute this cell? (yes/no): ")
if response.lower() == "yes":

    index = 0

    """
    0 = jan
    1 = feb
    2 = mar
    3 = apr
    4 = may
    5 = jun
    6 = jul
    7 = aug
    8 = sep
    9 = oct
    10 = nov
    11 = dec
    """

    # specify august onward
    for i in range(len(year_iter_2023)-1):
    # for i in range(8,9):

        j = 1 # iterator for names
        # j = 43 # make this the name iterator because of error
        month = 1

        for chunk in chunks:

            print(f"Test case: {index}")

            index = index + 1

            if chunk is None:
                continue

            print("________Sample of this chunk________")
            print(f"{chunk["coordinates"][0][0][1:5]}\n")

            """
            Begin time count
            """
            t_before = time.time()


            """
            Define our filters
            """
            # define current geometry filter
            geometry_filter = {
                "type": "GeometryFilter", # set filter type as geometry
                "field_name": "geometry", # give json column containing geometry
                "config": chunk # input json
            }


            # define date range, should be for a single year
            date_range_filter = {
                "type": "DateRangeFilter", 
                "field_name": "acquired",  # selects for when imagery was measured, not published
                "config": { 
                    "gte": f"{year_iter_2023[i]}-{month_nums[i]}-01T00:00:00.000Z", # greater then equal to
                    "lt":  f"{year_iter_2023[i+1]}-{month_nums[i+1]}-01T00:00:00.000Z" # less then
                }
            }
            print(f"gte: {year_iter_2023[i]}-{month_nums[i]}-01T00:00:00.000Z")
            print(f"lt: {year_iter_2023[i+1]}-{month_nums[i+1]}-01T00:00:00.000Z")


            # cloud filter
            cloud_cover_filter = {
                "type": "RangeFilter", # generalized filter type that takes range of values
                "field_name": "cloud_cover", # ask for cloud cover
                "config": {
                    "lte": 0.01 # filter for all scenes w <1% cloud cover
                }
            }

            asset_filter = {
                "type": "AndFilter",
                "config": [
                    {
                        "type": "AssetFilter",
                        "config": [
                            "ortho_analytic_4b"
                        ]
                    }
                ]
            }


            # combine filters
            combined_filters = {
                "type": "AndFilter", # filter type for and conditional
                "config": [geometry_filter, date_range_filter, cloud_cover_filter, asset_filter]
            }





            """
            Take our filters and make a search request
            """
            # defines package type
            # PSScene has orthorectified 8 and 4 band imagery with the udm2 file
            # you can take imagery from multiple types but we only need this one
            item_types = ["PSScene"]

            # feed our filters and package type selection into a filter dict
            search_request = {
                "item_types": item_types,
                "filter": combined_filters
            }

            chunked_error_cnt = 0

            while chunked_error_cnt < 5:
                try: 

                    # and we search Planet labs imagery with it
                    # requests version - sends new connection each time but tends to get chunkederrors
                    # search_result = \
                    #     requests.post( # post sends our request to https api
                    #         "https://api.planet.com/data/v1/quick-search",
                    #         auth = HTTPBasicAuth(planet_key, ''),  # we give it our key
                    #         json = search_request, # and our request dict
                    #         timeout=999
                    #     )
                    
                    # sessions version - maintains connection over requests
                    # allows retry logic, but may cause some unexpected behaviors, trying it for now
                    search_result = \
                        session.post( # post sends our request to https api
                            "https://api.planet.com/data/v1/quick-search",
                            auth = HTTPBasicAuth(planet_key, ''),  # we give it our key
                            json = search_request, # and our request dict
                            timeout=999
                        )
                    
                    break
                except requests.exceptions.ChunkedEncodingError:
                    if chunked_error_cnt < 5: 
                        print("Chunked encoding error on this chunk. Atempting again...")
                    else:
                        print(f"Max retries reached on chunked error handler. Restart ordering workflow on month {i} to try again...")
                    
                    chunked_error_cnt += 1


            # remove stray scenes without the ortho_analytic_4b_sr asset
            new_list = []
            error_cnt = 0

            while error_cnt <= 5:
                try: 
                    edit_this = search_result.json()
                    baseline_search = edit_this

                    for scene in enumerate(edit_this["features"]):
                        if "ortho_analytic_4b_sr" in scene[1]["assets"]:
                            new_list.append(scene[1])
                    if len(new_list) != len(edit_this["features"]):
                        edit_this["features"] = new_list
                        print(f"Index has scenes without ortho_analytic_4b_sr. Removed {len(baseline_search['features']) - len(new_list)} scenes...")
                    
                    break

                except json.JSONDecodeError:
                    if error_cnt < 5:
                        print("JSON parsing error in the stray scene removal step. Retrying...")
                    else:
                        print("JSON parsing error in the stray scene removal step met max retries. This chunk may fail...")

                    error_cnt += 1

                except Exception as e:
                    if error_cnt < 5:
                        print("Some other error occured. Attempting again...")
                    else:
                        print(f"Ordering failed in the stray scene removal due to some error, and reached max retries: {e}")

                    error_cnt += 1
                
                if error_cnt > 5:
                    break



            """
            Crunch all IDs into a list
            """
            ids = [feature['id'] for feature in edit_this["features"]]




            """
            Make our order request

            1. Setup our order json
            2. Setup our tools
            """
            # set content type to json
            headers = {"content-type": "application/json"}

            # init order parameters, this one has the 4 band
            product = [
                { 
                    "item_ids": ids, 
                    "item_type": "PSScene", 
                    "product_bundle": "analytic_sr_udm2", # ortho 4 band surface reflectance
                }
            ]


            # init clip to sb county
            clip =  {
                "clip": {
                    "aoi": chunk
                }
            }

            # init ndvi calculation
            bandmath = {
                "bandmath": {
                    "b1": "b1",
                    "b2": "b2", 
                    "b3": "b3", 
                    "b4": "b4", 
                    "b5": "(b4 - b3) / (b4 + b3)"
                }
            }

            # make name
            order_name = f"{var_name_as_str(chunk)}_month_{month_nums[i]}_year_2023_download_{j}_first_run"
            # order_name = "error_test"

            # create request json
            tool_request = { 
                "name": order_name, 
                "products": product, 
                "tools": [clip, bandmath], 
                "delivery": {"single_archive": True, "archive_type": "zip"}
            }

            # dry run version
            # tool_request = { 
            #     "name": order_name, 
            #     "products": product, 
            #     "tools": [clip, bandmath], 
            #     "delivery": None
            # }



            """
            Send in the order
            """
            # call the function
            
            retry_counter = 0

            order_url = place_order(tool_request, auth, retry_counter, j-1, month = j)
            print("\n")


            """
            Wait before sending in next order
            """
            # time.sleep(2)
            time.sleep(0.5)

            # and increment the order labeler
            j = j + 1



    # now we can poll for success
    poll_for_success(orders_api_url, auth)


    """
    End time count and output total time PARTIALLY DEPRECATED, MOVED TO OUTSIDE THE FOR LOOP
    """

    t_after = time.time()
    locals()[f"{chunk}_exec_t"] = t_after - t_before
    
    print(f"Total execution time for {order_name}: {locals()[f"{chunk}_exec_t"]} seconds")
        

#________________End of code chunk___________________#

    print("Cell executed!")
else:
    print("Execution canceled.")

Test case: 0
________Sample of this chunk________
[[-119.71458056841794, 34.961439751849944], [-119.71333829402823, 34.961436662342166], [-119.71334264534441, 34.960255882646365], [-119.7145849009689, 34.96025897207312]]

gte: 2023-01-01T00:00:00.000Z
lt: 2023-02-01T00:00:00.000Z
a5a9b395-7e40-4f7d-8874-a1152ce94ce3


Test case: 1
________Sample of this chunk________
[[-119.44832491961262, 34.80811384832367], [-119.4475158565708, 34.8081099686162], [-119.44752090357767, 34.80739646178308], [-119.44832995922324, 34.8074003414593]]

gte: 2023-01-01T00:00:00.000Z
lt: 2023-02-01T00:00:00.000Z
5c8ac9c0-a9be-432f-8ff6-dc486980de8e


Test case: 2
________Sample of this chunk________
[[-120.1535889694108, 34.66730556983106], [-120.15219088363207, 34.66730743374561], [-120.15218926503775, 34.66647630410193], [-120.1535873359328, 34.66647444019986]]

gte: 2023-01-01T00:00:00.000Z
lt: 2023-02-01T00:00:00.000Z
40a57fcf-80fa-47da-9280-e03ce97f77cc


Test case: 3
________Sample of this chunk________

### Downloading imagery

In [4]:
# ping planet
r = requests.get(orders_api_url, auth = auth)

# turn response into a json/dict
order_response = r.json()

# isolate state of most recent order
order_results = order_response["orders"][0]["state"]



all_scenes = []

first_scenes = [[squid["name"], squid["id"], squid["created_on"]] for squid in order_response["orders"]]
all_scenes.extend(first_scenes)

json_error_counter = 0
i = 0

def json_decode_error_handler(func, args):
    json_error_counter = 0
    result = None

    while json_error_counter < 10:
        try:
            result = func(*args)
            break
        except json.JSONDecodeError:
            json_error_counter = json_error_counter + 1
    
    if json_error_counter < 50:
        print(f"Completed successfully. Json decode errors: {json_error_counter}...") 
    elif json_error_counter >= 50:
        print("At max Json decode errors. Completed unsuccessfully. Returning Null...")

    return result


def scene_pagenation(order_response, scenes, depth): 
    scenes_iter = []
    retry_counter = 0

    try:
        for i in range(depth):
            if "_links" in order_response and "next" in order_response["_links"]:
                next_link = order_response["_links"]["next"]
                next_scenes_ping = session.get(next_link)
            else:
                print("No more pages available. Ending pagination.")
                break  # Prevent error when no next page


            time.sleep(0.5)

            if next_scenes_ping.status_code != 200:
                print(f"Error {next_scenes_ping.status_code}: {next_scenes_ping.text}")
            else:
                pag_json = next_scenes_ping.json()

            next_scenes = [
                [
                    squid["name"], 
                    squid["id"], 
                    squid["created_on"], 
                    squid["products"][0]["item_ids"][0]
                ] 
                    for squid in pag_json["orders"]
                ]

            scenes_iter.append(next_scenes)

            order_response = pag_json
            
    except Exception as e:
            print(f"Error during pagination at depth {i}: {e}. Retrying...")

            time.sleep(2 ** retry_counter)  # Exponential backoff
            retry_counter += 1

            scene_pagenation(order_response, scenes, depth)

            

        
    for chunk in scenes_iter:
        scenes.extend(chunk)

    return scenes

all_scenes = json_decode_error_handler(
    scene_pagenation, [order_response, all_scenes, 9999]
)





Error during pagination at depth 643: Response ended prematurely. Retrying...
Error during pagination at depth 125: Response ended prematurely. Retrying...
Error during pagination at depth 146: Response ended prematurely. Retrying...
Error during pagination at depth 184: Response ended prematurely. Retrying...
No more pages available. Ending pagination.
Completed successfully. Json decode errors: 0...


In [25]:
month_sea = ["month_12", "month_11", "month_10", "month_09", "month_08", "month_07", "month_06", "month_05", "month_04", "month_03", "month_02", "month_01"]
month_name = ["dec", "nov", "oct", "sep", "aug", "jul", "jun", "may", "apr", "mar", "feb", "jan"]
year_sea = ["2019", "2020", "2021", "2022", "2023"]


In [ ]:
for m_search, m_name in zip(month_sea, month_name):

    append_list = []

    for scene in all_scenes:
        if m_search in scene[0] and year_sea[3] in scene[0] and "run" in scene[0] and "dry_run" not in scene[0]:
            append_list.append(scene)

    globals()[f"all_{m_name}_{year_sea[3]}_prelim"] = append_list
    

In [ ]:
# all_jan_2020_scenes = [scene[1] for scene in all_jan_2020_prelim if "third_run" in scene[0]]
# all_feb_2020_scenes = [scene[1] for scene in all_feb_2020_prelim]
# all_mar_2020_scenes = [scene[1] for scene in all_mar_2020_prelim]
# all_apr_2020_scenes = [scene[1] for scene in all_apr_2020_prelim]
# all_may_2020_scenes = [scene[1] for scene in all_may_2020_prelim if "fourth_run" in scene[0]]
# all_jun_2020_scenes = [scene[1] for scene in all_jun_2020_prelim if "fifth_run" in scene[0]]
# all_jul_2020_scenes = [scene[1] for scene in all_jul_2020_prelim]
# all_aug_2020_scenes = [scene[1] for scene in all_aug_2020_prelim]
# all_sep_2020_scenes = [scene[1] for scene in all_sep_2020_prelim]
# all_oct_2020_scenes = [scene[1] for scene in all_oct_2020_prelim]
# all_nov_2020_scenes = [scene[1] for scene in all_nov_2020_prelim]
# all_dec_2020_scenes = [scene[1] for scene in all_dec_2020_prelim]
# len(all_dec_2020_scenes)

# download_list = [all_jan_2020_scenes, 
#                  all_feb_2020_scenes, 
#                  all_mar_2020_scenes, 
#                  all_apr_2020_scenes, 
#                  all_may_2020_scenes, 
#                  all_jun_2020_scenes, 
#                  all_jul_2020_scenes, 
#                  all_aug_2020_scenes, 
#                  all_sep_2020_scenes, 
#                  all_oct_2020_scenes, 
#                  all_nov_2020_scenes, 
#                  all_dec_2020_scenes]

In [ ]:
# all_jan_2021_scenes = [scene[1] for scene in all_jan_2021_prelim]
# all_feb_2021_scenes = [scene[1] for scene in all_feb_2021_prelim if "fourth_run" in scene[0]]
# all_mar_2021_scenes = [scene[1] for scene in all_mar_2021_prelim if "fourth_run" in scene[0]]
# all_apr_2021_scenes = [scene[1] for scene in all_apr_2021_prelim if "ninth_run" in scene[0]]
# all_may_2021_scenes = [scene[1] for scene in all_may_2021_prelim]
# all_jun_2021_scenes = [scene[1] for scene in all_jun_2021_prelim if "ninth_run" in scene[0]]
# all_jul_2021_scenes = [scene[1] for scene in all_jul_2021_prelim]
# all_aug_2021_scenes = [scene[1] for scene in all_aug_2021_prelim if "tenth_run" in scene[0]]
# all_sep_2021_scenes = [scene[1] for scene in all_sep_2021_prelim]
# all_oct_2021_scenes = [scene[1] for scene in all_oct_2021_prelim]
# all_nov_2021_scenes = [scene[1] for scene in all_nov_2021_prelim]
# all_dec_2021_scenes = [scene[1] for scene in all_dec_2021_prelim]

# download_list = []
# for month in month_name:
#     download_list.append(globals()[f"all_{month}_2021_scenes"])

In [9]:
# all_jan_2022_scenes = [scene[1] for scene in all_jan_2022_prelim]
# all_feb_2022_scenes = [scene[1] for scene in all_feb_2022_prelim]
# all_mar_2022_scenes = [scene[1] for scene in all_mar_2022_prelim]
# all_apr_2022_scenes = [scene[1] for scene in all_apr_2022_prelim]
# all_may_2022_scenes = [scene[1] for scene in all_may_2022_prelim if "second_run" in scene[0]]
# all_jun_2022_scenes = [scene[1] for scene in all_jun_2022_prelim]
# all_jul_2022_scenes = [scene[1] for scene in all_jul_2022_prelim]
# all_aug_2022_scenes = [scene[1] for scene in all_aug_2022_prelim]
# all_sep_2022_scenes = [scene[1] for scene in all_sep_2022_prelim]
# all_oct_2022_scenes = [scene[1] for scene in all_oct_2022_prelim]
# all_nov_2022_scenes = [scene[1] for scene in all_nov_2022_prelim]
# all_dec_2022_scenes = [scene[1] for scene in all_dec_2022_prelim]

# for name in month_name:
#     print(name)
#     print(f"length: {len(globals()[f"all_{name}_2022_scenes"])}")
#     print(f"sample: {globals()[f"all_{name}_2022_prelim"][115]}\n")


# download_list = []
# for month in month_name:
#     download_list.append(globals()[f"all_{month}_2022_scenes"])

    

dec
length: 341
sample: ['chunk_month_12_year_2022_download_227_second_run', '43a448eb-fdd5-43da-8043-c79823fa89e8', '2025-05-06T08:41:00.800782Z', '20221227_174400_23_2442']

nov
length: 341
sample: ['chunk_month_11_year_2022_download_227_second_run', '605e9533-ed46-4179-b368-d5967b0fb382', '2025-05-06T08:29:39.123317Z', '20221129_183548_35_240c']

oct
length: 340
sample: ['chunk_month_10_year_2022_download_226_second_run', '94c58fa1-ce0d-42b3-99e9-246bf3f8a8bf', '2025-05-06T08:17:48.692252Z', '20221027_182424_52_2486']

sep
length: 341
sample: ['chunk_month_09_year_2022_download_227_second_run', '8c11c650-114a-48f9-9e7a-7d96ac99eb14', '2025-05-06T08:06:34.775381Z', '20220929_182029_44_2495']

aug
length: 341
sample: ['chunk_month_08_year_2022_download_227_second_run', '37192820-5adb-4c66-916f-3e0b501d2cf9', '2025-05-06T07:54:59.470934Z', '20220830_181703_48_222f']

jul
length: 340
sample: ['chunk_month_07_year_2022_download_227_second_run', 'a7ed3422-6498-493f-ab3e-95734e79f6a5', '20

In [ ]:
# all_jan_2023_scenes = [scene[1] for scene in all_jan_2023_prelim]
# all_feb_2023_scenes = [scene[1] for scene in all_feb_2023_prelim]
# all_mar_2023_scenes = [scene[1] for scene in all_mar_2023_prelim]
# all_apr_2023_scenes = [scene[1] for scene in all_apr_2023_prelim]
# all_may_2023_scenes = [scene[1] for scene in all_may_2023_prelim]
# all_jun_2023_scenes = [scene[1] for scene in all_jun_2023_prelim]
# all_jul_2023_scenes = [scene[1] for scene in all_jul_2023_prelim]
# all_aug_2023_scenes = [scene[1] for scene in all_aug_2023_prelim]
# all_sep_2023_scenes = [scene[1] for scene in all_sep_2023_prelim]
# all_oct_2023_scenes = [scene[1] for scene in all_oct_2023_prelim]
# all_nov_2023_scenes = [scene[1] for scene in all_nov_2023_prelim]
# all_dec_2023_scenes = [scene[1] for scene in all_dec_2023_prelim]


# for name in month_name:
#     print(name)
#     print(f"length: {len(globals()[f"all_{name}_2023_scenes"])}")
#     print(f"sample: {globals()[f"all_{name}_2023_prelim"][115]}\n")
#     # [scene for scene in all_feb_2023_prelim]

# download_list = []
# for month in month_name:
#     download_list.append(globals()[f"all_{month}_2023_scenes"])

dec
length: 327
sample: ['chunk_month_12_year_2023_download_218_first_run', '1ffb4602-6f8f-4bc8-8fb2-aa5127a8dfbf', '2025-05-06T11:36:54.161402Z', '20231224_184008_90_247c']

nov
length: 334
sample: ['chunk_month_11_year_2023_download_220_first_run', '011c876e-8cc2-47aa-b04c-27fd85740c46', '2025-05-06T11:17:52.519131Z', '20231128_175511_03_24d0']

oct
length: 334
sample: ['chunk_month_10_year_2023_download_220_first_run', '5d99b1a6-f256-41b8-93bb-0ef7df455bf7', '2025-05-06T11:05:12.002278Z', '20231013_183622_60_2478']

sep
length: 325
sample: ['chunk_month_09_year_2023_download_220_first_run', 'beb011c9-1ac8-4190-a3ee-b0691376496c', '2025-05-06T10:53:46.678522Z', '20230926_175334_55_24b3']

aug
length: 331
sample: ['chunk_month_08_year_2023_download_217_first_run', 'd62fded8-0037-42db-94d9-29ece946776b', '2025-05-06T10:40:14.945668Z', '20230830_175255_88_2464']

jul
length: 334
sample: ['chunk_month_07_year_2023_download_220_first_run', 'e4a02df5-5a50-43f4-bd25-5181e6f46045', '2025-05-

In [13]:
pl = Planet(session=Session(auth=APIKeyAuth(key=planet_key)))

# NOTE, 2020 FOLDER IS BACKWARDS, should now be fixed?

# REMEMBER TO RESET MONTH FOLDER BEFORE RUNNING

# safety check, only run cell if you specifically say "yes"
response = input("Are you sure you want to execute this cell? (yes/no): ")
if response.lower() == "yes":
            
    except_counter = 0
    sleep_counter = 1

    for download_month, m_name in zip(download_list, month_name):

        for scene in download_month:
            except_counter = 0
            sleep_counter = 1
            while True:
                try: 
                    pl.orders.download_order(
                        scene, # get order id
                        overwrite = False, # don't overwrite files that already exist
                        directory = f"/data/wildfire_prep/full_planet/raw_zip/2023/{m_name}" # set directory to our data folder in workbench-2
                        )
                    break

                except Exception as e:
                    if except_counter < 5:
                        print(f"Error at {scene}: {e}. Retrying...")
                        except_counter += 1 
                        
                        time.sleep(sleep_counter)
                        sleep_counter = sleep_counter * 2
                        print(f"Waiting for {sleep_counter} seconds before attempting again...")

    print("Cell executed!")
    
else:
    print("Execution canceled.")

Error at 5849ddd6-2752-41fc-b4fa-5b0920a04eb2: <html>
<head><title>504 Gateway Time-out</title></head>
<body>
<center><h1>504 Gateway Time-out</h1></center>
<hr><center>nginx</center>
</body>
</html>
. Retrying...
Waiting for 2 seconds before attempting again...
Cell executed!


##### Unzip the imagery

In [46]:
extract_path = "/data/wildfire_prep/full_planet/raw_zip"
root_export_path = "/data/wildfire_prep/full_planet/unzipped"

test_list = []
for year in year_sea:
    for month in month_name: 
        for root,dirs,files in os.walk(extract_path, topdown=True):
            if "output.zip" in files and f'{year}/' in root and f'{month}/' in root:
                print(f"{root[32:]}/{files[1]}")
                # # print(dirs)
                # print(files[1])
                # print('--------------------------------')
                test_list.append(f"{root}/{files[1]}")
len(test_list)


raw_zip/2019/dec/dd4d204f-94d2-4467-be26-e752a183ffd3/output.zip
raw_zip/2019/dec/96b37443-bc95-4e46-9229-b06b65edb426/output.zip
raw_zip/2019/dec/eac7010b-aef8-4ae9-af10-649eb291b615/output.zip
raw_zip/2019/dec/0702c8ac-490f-4cc8-ba1f-34802b8922b9/output.zip
raw_zip/2019/dec/98cabc46-d769-4e5a-9b39-c7d869958261/output.zip
raw_zip/2019/dec/8ef4b04d-ad68-4347-8470-69f72840efdd/output.zip
raw_zip/2019/dec/fa64deca-1e0d-45fe-a672-5e34d68d0dde/output.zip
raw_zip/2019/dec/1bb6a025-5bc3-419d-aa0e-50b46854e0bf/output.zip
raw_zip/2019/dec/da04795c-0bfb-4c40-8f06-684a47ffbab3/output.zip
raw_zip/2019/dec/694864ae-b331-430d-bcb8-1e0b09a13f49/output.zip
raw_zip/2019/dec/7394b303-98dc-4b54-b33d-3a9636017575/output.zip
raw_zip/2019/dec/46ad2cdf-8686-4f1c-952e-aae912b7c63c/output.zip
raw_zip/2019/dec/5d7cd4cd-9b68-4458-86cc-04ef18fae097/output.zip
raw_zip/2019/dec/74c8180d-eb59-4ace-b3ea-5e887647d153/output.zip
raw_zip/2019/dec/39f921aa-b77d-4d58-9285-c85d264408be/output.zip
raw_zip/2019/dec/fe16c5b7

20535

In [48]:
# path = "/data/wildfire_prep/full_planet/raw_zip/2019/nov/9c522f12-f219-4221-85b6-1e1e661bd603"
# safety check, only run cell if you specifically say "yes"
response = input("Are you sure you want to execute this cell? (yes/no): ")
if response.lower() == "yes":

    for year in year_sea:
        for month in month_name: 
            for root,dirs,files in os.walk(extract_path, topdown=True):
                if "output.zip" in files and f'{year}/' in root and f'{month}/' in root:
                    with ZipFile(f"{root}/output.zip", 'r') as zip: 

                        # zip.printdir()

                        print(f"Unzipping {root[32:]}/{files[1]} to unzipped/{year}/{month}")

                        # extracting all the files 
                        print('Extracting all the files now...') 

                        zip.extractall(path = f"{root_export_path}/{year}/{month}") 
                        
                        print('Done!') 
                        print("---------------------------\n")



    print("Cell executed!")
else:
    print("Execution canceled.")

Unzipping raw_zip/2019/dec/dd4d204f-94d2-4467-be26-e752a183ffd3/output.zip to unzipped/2019/dec
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2019/dec/96b37443-bc95-4e46-9229-b06b65edb426/output.zip to unzipped/2019/dec
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2019/dec/eac7010b-aef8-4ae9-af10-649eb291b615/output.zip to unzipped/2019/dec
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2019/dec/0702c8ac-490f-4cc8-ba1f-34802b8922b9/output.zip to unzipped/2019/dec
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2019/dec/98cabc46-d769-4e5a-9b39-c7d869958261/output.zip to unzipped/2019/dec
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2019/dec/8ef4b04d-ad68-4347-8470-69f72840efdd/output.zip to unzipped/2019/dec
Extracting all the files now...
Done!
---------------------------

Unzipping raw_zip/2019